In [0]:
# Bibliotecas
import re
from pyspark.sql import functions as F

# CONFIGURAÇÕES
CAMINHO_ORIGEM = "/Volumes/mba/stage/dados_bruto/IBGE"
TABELA_BRASIL_UFS = "mba.raw.ibge_populacao_brasil_ufs"
TABELA_MUNICIPIOS = "mba.raw.ibge_populacao_municipios"

In [0]:
# ============================================
# LISTAGEM DOS ARQUIVOS
# ============================================

arquivos = dbutils.fs.ls(CAMINHO_ORIGEM)

arquivos_ibge = [
    arquivo
    for arquivo in arquivos
    if re.match(
        r"^IBGE-POP\d{4}_.*\.xls",
        arquivo.name,
        re.IGNORECASE
    )
]

if len(arquivos_ibge) == 0:
    dbutils.notebook.exit("Nenhum arquivo IBGE encontrado.")

print(f"Arquivos IBGE encontrados: {len(arquivos_ibge)}")

In [0]:
from pyspark.sql import functions as F


# ============================================================
# FUNÇÃO PARA CARREGAR A POPULAÇÃO DO IBGE
# ============================================================

def carregar_populacao_ibge(caminho_arquivo, arq, ano_referencia):

    # LÊ O ARQUIVO EXCEL
    if arq == "IBGE-POP2017_20220905.xls":
        df_origem = (
            spark.read
            .option("headerRows", 1)
            .option("dataAddress", "BRASIL_E_UFs!A2:C100")
            .excel(caminho_arquivo)
        )
    else:
        df_origem = (
            spark.read
            .option("headerRows", 1)
            .option("dataAddress", "BRASIL E UFs!A2:C100")
            .excel(caminho_arquivo)
        )

    # MANTÉM E RENOMEIA AS COLUNAS NECESSÁRIAS
    df_destino = (
        df_origem
        .select(            
            # Ano extraído do nome do arquivo
            F.lit(ano_referencia)
            .cast("int")
            .alias("AnoReferencia"),

            # Brasil / Região / Unidade da Federação
            F.col("BRASIL E UNIDADES DA FEDERAÇÃO")
            .cast("string")
            .alias("BrasilUnidadeFederacao"),

            # População estimada
            F.regexp_replace(
                F.col("POPULAÇÃO ESTIMADA"),
                r"[^0-9]",
                ""
            )
            .cast("bigint")
            .alias("PopulacaoEstimada"),

            # Nome do arquivo de origem
            F.lit(arq)
            .cast("string")
            .alias("NomeArquivo"),

            # Data e hora da carga
            F.current_timestamp()
            .alias("DataCarga")
        )

        # Remove registros sem Brasil / Região / UF
        .filter(
            F.col("BrasilUnidadeFederacao").isNotNull()
        )

        # Remove registros pupulacao estimada
        .filter(
            F.col("PopulacaoEstimada").isNotNull()
        )
    )

    # ========================================
    # VALIDA SE O ARQUIVO JÁ FOI CARREGADO
    # ========================================

    arquivo_ja_carregado_brasiluf = (
        spark.table(TABELA_BRASIL_UFS)
        .filter(
            F.col("NomeArquivo") == nome_arquivo
        )
        .limit(1)
        .count()
    )

     # ========================================
    # GRAVAÇÃO - MUNICÍPIOS
    # ========================================

    if arquivo_ja_carregado_brasiluf == 0:

        quantidade_UFS = (df_destino.count())

        df_destino.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(TABELA_BRASIL_UFS)

        print(f"Planilha: BRASIL_E_UFs: "
                f"{quantidade_UFS} registros"
        )


In [0]:
def carregar_municipio_ibge(caminho_arquivo, arq, ano_referencia):

    # LÊ O ARQUIVO EXCEL
    df_origem = (
        spark.read
        .option("headerRows", 1)
        .option("dataAddress", "Municípios!A2:e1000")
        .excel(caminho_arquivo)
        )
    
     # ========================================
    # TRANSFORMAÇÃO - MUNICÍPIOS
    # ========================================
    colunas = (df_origem.columns)
    df_municipios = (
        df_origem
        .select(
            F.col(f"`{colunas[0]}`").alias("SiglaUF"),
            F.col(f"`{colunas[1]}`").alias("CodigoUF"),
            F.col(f"`{colunas[2]}`").alias("CodigoMunicipio"),
            F.col(f"`{colunas[3]}`").alias("NomeMunicipio"),
            F.col(f"`{colunas[4]}`").alias("PopulacaoEstimada")
        )

        # Remove linhas sem município
        .filter(
            F.col(
                "NomeMunicipio"
            ).isNotNull()
        )

        # ------------------------------------
        # CÓDIGO DO MUNICÍPIO
        #
        # Remove possível .0 do Excel
        # e completa com zeros à esquerda.
        # ------------------------------------

        .withColumn(
            "CodigoMunicipio",

            F.lpad(
                F.regexp_replace(
                    F.col(
                        "CodigoMunicipio"
                    ),
                    r"\.0$",
                    ""
                ),
                5,
                "0"
            )
        )

        # ------------------------------------
        # CÓDIGO DA UF
        # ------------------------------------

        .withColumn(
            "CodigoUF",

            F.regexp_replace(
                F.col("CodigoUF"),
                r"\.0$",
                ""
            )
        )

        # ------------------------------------
        # POPULAÇÃO
        # ------------------------------------

        .withColumn(
            "PopulacaoEstimada",
            F.regexp_replace(
                F.trim(F.col("PopulacaoEstimada")),
                r"[^0-9]",
                ""
            )
        )

        # Ano de referência
        .withColumn(
            "AnoReferencia",
            F.lit(ano_referencia)
        )

        # Nome do arquivo
        .withColumn(
            "NomeArquivo",
            F.lit(arq)
        )

        # Data/hora da carga
        .withColumn(
            "DataCarga",
            F.current_timestamp()
        )

        # Ordem final
        .select(
            "AnoReferencia",
            "SiglaUF",
            "CodigoUF",
            "CodigoMunicipio",
            "NomeMunicipio",
            "PopulacaoEstimada",
            "NomeArquivo",
            "DataCarga"
        )
    )

    # ========================================
    # VALIDA SE O ARQUIVO JÁ FOI CARREGADO
    # ========================================

    arquivo_ja_carregado_municipios = (
        spark.table(TABELA_MUNICIPIOS)
        .filter(
            F.col("NomeArquivo") == nome_arquivo
        )
        .limit(1)
        .count()
    )
        
    # ========================================
    # GRAVAÇÃO - MUNICÍPIOS
    # ========================================

    if arquivo_ja_carregado_municipios == 0:

        quantidade_municipios = (df_municipios.count())

        df_municipios.write \
            .mode("append") \
            .saveAsTable(TABELA_MUNICIPIOS)

        print(f"Planilha: Municípios: "
                f"{quantidade_municipios} registros"
        )

In [0]:
# Limpa tabela
spark.sql(f"truncate table {TABELA_BRASIL_UFS}")
spark.sql(f"truncate table {TABELA_MUNICIPIOS}")


for arquivo in arquivos_ibge:
    caminho_arquivo = arquivo.path.replace("dbfs:", "")
    nome_arquivo = arquivo.name
    print(f"Arquivo: {nome_arquivo}")

    # EXTRAI O NOME DO ARQUIVO
    arq = caminho_arquivo.split("/")[-1]

    # EXTRAI O ANO DE REFERÊNCIA DO NOME DO ARQUIVO
    ano_referencia = int(
        arq
        .replace("IBGE-POP", "")
        .split("_")[0]
    )

    # chama função para carregar população
    carregar_populacao_ibge(caminho_arquivo, arq, ano_referencia)

    # chama função para carregar população
    carregar_municipio_ibge(caminho_arquivo, arq, ano_referencia)
    print("=" * 40)

    
print("=" * 70)
print("PROCESSAMENTO FINALIZADO")


In [0]:
dbutils.notebook.exit("OK")